In [1]:
import os
import json

In [2]:
def read_jsonl(path):
    with open(path, "r") as f:
        data = [json.loads(ex) for ex in f]
    return data

In [4]:
data = read_jsonl("../../data/semcoder/final_dataset_w_concepts_and_thoughts.jsonl")

In [7]:
for ex in data:
    print(ex.keys())
    print(ex['concepts'])
    print(ex)
    break

dict_keys(['user_prompt', 'assistant_prompt', 'refcode', 'reasoning_content', 'content', 'idx', 'task_type', 'concepts'])
{'for_statement': [[85, 115]], 'while_statement': [], 'comparison_operator': [], 'boolean_operator': [], 'not_operator': [], 'call': [[127, 152], [166, 197], [233, 303]], 'assignment': [[59, 80]], 'list_comprehension': [], 'augmented_assignment': [], 'unary_operator': [], 'binary_operator': [], 'subscript': [], 'pattern_list': [], 'string': [[259, 266], [268, 276], [278, 287], [289, 295], [298, 302]]}
{'user_prompt': '\nSimulate the Execution : You are given a Python function and an assertion containing a\nfunction input . Complete the assertion containing the execution output corresponding to\nthe given input in [ ANSWER ] and [/ ANSWER ] tags .\n[PYTHON]\ndef filter_strings_by_prefix(list_of_strings, prefix):\n    filtered_strings = []\n    for string in list_of_strings:\n        if string.startswith(prefix):\n            filtered_strings.append(string)\n    retur

In [8]:
from __future__ import annotations
import json
import random
from collections import Counter, defaultdict
from typing import Iterable

In [9]:
def read_jsonl(path: str | Path) -> list[dict]:
    """Read a JSONL file into a list of dicts. Skips blank lines, reports parse errors with line numbers."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")
 
    records = []
    with path.open("r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Bad JSON at {path}:{lineno}: {e}") from e
    return records
 
 
def iter_jsonl(path: str | Path) -> Iterator[dict]:
    """Streaming reader — use when the file doesn't fit in memory comfortably."""
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"Bad JSON at {path}:{lineno}: {e}") from e
 
 
def write_jsonl(records: Iterable[dict], path: str | Path) -> None:
    """Write a list of dicts to a JSONL file (one JSON object per line)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
 
 
# ---------- helpers ----------
 
def get_present_concepts(example: dict) -> set[str]:
    """Return the set of concepts that actually appear (non-empty list) in an example."""
    concepts = example.get("concepts", {}) or {}
    return {name for name, spans in concepts.items() if spans}
 
 
def concept_counts(examples: Iterable[dict]) -> Counter:
    """Count how many examples contain each concept."""
    c = Counter()
    for ex in examples:
        c.update(get_present_concepts(ex))
    return c
 
 
def imbalance(counts: Counter) -> dict:
    """Summary stats describing how balanced a sample is."""
    if not counts:
        return {"min": 0, "max": 0, "spread": 0, "stdev": 0.0}
    vals = list(counts.values())
    mean = sum(vals) / len(vals)
    var = sum((v - mean) ** 2 for v in vals) / len(vals)
    return {
        "min": min(vals),
        "max": max(vals),
        "spread": max(vals) - min(vals),
        "stdev": var ** 0.5,
    }
 

In [10]:
def greedy_balanced_sample(
    dataset: list[dict],
    n_samples: int,
    seed: int = 0,
) -> list[dict]:
    """
    Greedy: at each step, pick the example that best helps the most
    under-represented concept. Ties broken by a seeded shuffle.
 
    Score for a candidate example given current counts:
      score = sum( 1 / (1 + current_count[c]) for c in example's concepts )
    Rare concepts get higher weight, so examples covering them are preferred.
    """
    rng = random.Random(seed)
    indices = list(range(len(dataset)))
    rng.shuffle(indices)  # randomize tie-breaking order
 
    # Pre-compute concept sets once (avoid recomputing in the inner loop)
    present = [get_present_concepts(ex) for ex in dataset]
 
    # Universe of concepts that appear at least once
    all_concepts: set[str] = set().union(*present) if present else set()
    counts: Counter = Counter({c: 0 for c in all_concepts})
 
    chosen_idx: list[int] = []
    chosen_set: set[int] = set()
 
    target = min(n_samples, len(dataset))
    for _ in range(target):
        best_score = -1.0
        best_i = None
        for i in indices:
            if i in chosen_set:
                continue
            ex_concepts = present[i]
            if not ex_concepts:
                score = 0.0
            else:
                score = sum(1.0 / (1 + counts[c]) for c in ex_concepts)
            if score > best_score:
                best_score = score
                best_i = i
        if best_i is None:
            break
        chosen_idx.append(best_i)
        chosen_set.add(best_i)
        for c in present[best_i]:
            counts[c] += 1
 
    return [dataset[i] for i in chosen_idx]
 

In [11]:
def random_sample(dataset: list[dict], n_samples: int, seed: int = 0) -> list[dict]:
    """Plain random sampling — included as a baseline for comparison."""
    rng = random.Random(seed)
    return rng.sample(dataset, min(n_samples, len(dataset)))

In [13]:
def per_concept_quota_sample(
    dataset: list[dict],
    per_concept: int,
    seed: int = 0,
) -> list[dict]:
    """
    For each concept, ensure at least `per_concept` examples containing it
    are in the sample. Useful when you want a hard floor per concept.
 
    Because examples are multi-labeled, picking one example often satisfies
    multiple concept quotas at once, so the final size is typically much
    smaller than (per_concept * num_concepts).
    """
    rng = random.Random(seed)
    present = [get_present_concepts(ex) for ex in dataset]
    all_concepts: set[str] = set().union(*present) if present else set()
 
    # Index: concept -> shuffled list of example indices containing it
    by_concept: dict[str, list[int]] = defaultdict(list)
    for i, s in enumerate(present):
        for c in s:
            by_concept[c].append(i)
    for c in by_concept:
        rng.shuffle(by_concept[c])
 
    counts: Counter = Counter({c: 0 for c in all_concepts})
    chosen: set[int] = set()
 
    progress = True
    while progress:
        progress = False
        # Fill the most under-served concept first
        for c in sorted(all_concepts, key=lambda x: counts[x]):
            if counts[c] >= per_concept:
                continue
            for idx in by_concept[c]:
                if idx not in chosen:
                    chosen.add(idx)
                    for cc in present[idx]:
                        counts[cc] += 1
                    progress = True
                    break
 
    return [dataset[i] for i in sorted(chosen)]

In [14]:
def print_report(name: str, counts: Counter, total_examples: int) -> None:
    stats = imbalance(counts)
    print(f"\n{name}  (n={total_examples})")
    print("  per-concept counts (sorted):")
    for c, v in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"    {c:<24} {v}")
    print(f"  min={stats['min']}  max={stats['max']}  "
          f"spread={stats['spread']}  stdev={stats['stdev']:.2f}")

In [15]:
def make_synthetic_dataset(n: int = 500, seed: int = 42) -> list[dict]:
    rng = random.Random(seed)
    probs = {
        "call": 0.85, "assignment": 0.80, "for_statement": 0.55,
        "string": 0.50, "comparison_operator": 0.45, "binary_operator": 0.40,
        "subscript": 0.35, "boolean_operator": 0.25, "while_statement": 0.15,
        "augmented_assignment": 0.15, "not_operator": 0.12,
        "unary_operator": 0.10, "list_comprehension": 0.08, "pattern_list": 0.04,
    }
    data = []
    for i in range(n):
        concepts = {c: ([[0, 1]] if rng.random() < p else []) for c, p in probs.items()}
        data.append({"idx": i, "task_type": "output_pred", "concepts": concepts})
    return data
 

In [24]:
concepts = {}
for ex in data:
    for con in ex['concepts']:
        concepts[con] = concepts.get(con, 0) + (1 if ex['concepts'][con] else 0)

In [25]:
concepts

{'for_statement': 27777,
 'while_statement': 2850,
 'comparison_operator': 48954,
 'boolean_operator': 6479,
 'not_operator': 4126,
 'call': 60967,
 'assignment': 47637,
 'list_comprehension': 12989,
 'augmented_assignment': 14918,
 'unary_operator': 9002,
 'binary_operator': 37329,
 'subscript': 23466,
 'pattern_list': 9132,
 'string': 27014}

In [21]:

sampled = greedy_balanced_sample(data, n_samples=10000, seed=42)
print(concept_counts(sampled))
print(imbalance(concept_counts(sampled)))

KeyboardInterrupt: 

In [ ]:
def main(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser(description="Balanced sampling for a JSONL dataset.")
    p.add_argument("--input", "-i", help="Path to input .jsonl file")
    p.add_argument("--output", "-o", help="Path to write sampled .jsonl (optional)")
    p.add_argument(
        "--strategy", "-s",
        choices=["greedy", "quota", "random"],
        default="greedy",
        help="Sampling strategy (default: greedy)",
    )
    p.add_argument("--n", type=int, default=1000,
                   help="Sample size for greedy/random (default: 1000)")
    p.add_argument("--per-concept", type=int, default=50,
                   help="Floor per concept for quota strategy (default: 50)")
    p.add_argument("--seed", type=int, default=0, help="Random seed (default: 0)")
    p.add_argument("--demo", action="store_true",
                   help="Run on synthetic data instead of reading --input")
    args = p.parse_args(argv)
 
    # Load dataset
    if args.demo or not args.input:
        if not args.demo:
            print("No --input given; running on synthetic demo data.\n"
                  "Pass --input path/to/data.jsonl to use your own.\n",
                  file=sys.stderr)
        dataset = make_synthetic_dataset(n=500)
        source_label = "synthetic demo data"
    else:
        dataset = read_jsonl(args.input)
        source_label = args.input
 
    print(f"Loaded {len(dataset)} records from {source_label}")
    full_counts = concept_counts(dataset)
    print_report("FULL DATASET", full_counts, len(dataset))
 
    # Sample
    if args.strategy == "greedy":
        sampled = greedy_balanced_sample(dataset, args.n, seed=args.seed)
        label = f"Greedy balanced (target n={args.n})"
    elif args.strategy == "random":
        sampled = random_sample(dataset, args.n, seed=args.seed)
        label = f"Random (n={args.n})"
    else:  # quota
        sampled = per_concept_quota_sample(dataset, args.per_concept, seed=args.seed)
        label = f"Per-concept quota (>={args.per_concept} each)"
 
    print_report(label, concept_counts(sampled), len(sampled))
 
    # Optionally write out
    if args.output:
        write_jsonl(sampled, args.output)
        print(f"\nWrote {len(sampled)} records to {args.output}")
 
    return 0
 
 
if __name__ == "__main__":
    sys.exit(main())
 

In [ ]:
data = read_jsonl('../../data/combined/final_dataset_w_thinking_mode_concepts_task_type_and_rejected_trial_5.jsonl')

In [3]:
data = read_jsonl('../../data/combined/final_dataset_w_thinking_mode_concepts_task_type_and_rejected_trial_5.jsonl')

In [7]:
for ex in data[-5:]:
    # print(ex.keys())
    print(ex['metadata']['task_type'])
    print(ex.keys())
    print(ex['prompt'])
    print("^" * 20)
    print(ex['chosen'])
    print("-" * 10)
    print(ex['rejected'])
    print("*" * 20)

output_pred
dict_keys(['prompt', 'chosen', 'rejected', 'metadata'])

Simulate the Execution : You are given a Python function and an assertion containing a
function input . Complete the assertion containing the execution output corresponding to
the given input in [ ANSWER ] and [/ ANSWER ] tags .
[PYTHON]
def filter_strings_by_prefix(list_of_strings, prefix):
    filtered_strings = []
    for string in list_of_strings:
        if string.startswith(prefix):
            filtered_strings.append(string)
    return filtered_strings
assert filter_strings_by_prefix(["apple", "banana", "apricot", "kiwi"], "ap") == ??
[/PYTHON]


^^^^^^^^^^^^^^^^^^^^
### Understanding the Function
The function `filter_strings_by_prefix` is designed to filter a list of strings and return only those that start with a specified prefix. Let's break down the function:
- It takes two parameters: `list_of_strings`, which is a list of strings, and `prefix`, which is the string prefix we want to filter by.
- It initiali